# LLM Production Optimization

## Overview

Master the art of deploying LLMs cost-effectively and efficiently in production. Learn caching, batching, monitoring, and optimization strategies.

**Part of:** Phase 8 - MLOps

**Prerequisites:**
- Prompt Engineering (Phase 10)
- Local LLMs (Phase 13)
- Basic MLOps

**Outcome:** Deploy optimized, cost-effective LLM applications

---

## What You'll Learn

### Cost Optimization

- Token usage tracking and reduction
- Semantic caching strategies
- Batch processing for efficiency
- Model selection (cost vs. capability)
- Prompt compression techniques
- Fallback strategies (cheap → expensive)

### Performance Optimization

- Response time optimization
- Streaming responses
- Parallel processing
- Prefetching and speculation
- Edge deployment
- CDN for static responses

### Monitoring & Observability

- LLM metrics (latency, tokens, cost)
- Quality monitoring
- Error tracking
- User feedback loops
- A/B testing
- Cost alerts

### Infrastructure

- Load balancing
- Auto-scaling
- Rate limiting
- Circuit breakers
- Retry strategies
- Fallback models

In [ ]:
# Required packages (uncomment to install)
# !pip install openai redis tiktoken

---

## 1. Semantic Caching

Semantic caching uses embedding similarity to serve cached responses for queries that are
semantically equivalent - even if the wording is different. This is distinct from provider-level
prompt caching (covered in `09_llm_infrastructure.ipynb`), which reuses KV-cache at the token level.

**How it works:** Hash the first few dimensions of the query embedding to create a cache key.
Similar queries map to the same bucket and get a cache hit.

In [ ]:
import hashlib
import os


def get_embedding_hash(text: str, dimensions: int = 10) -> str:
    """Create a semantic hash for similar queries.

    Uses the first *dimensions* components of the embedding vector.
    In production, use a vector DB similarity search instead of hashing.
    """
    # Placeholder: simulate an embedding with a deterministic hash.
    # In production, replace with:
    #   from openai import OpenAI
    #   client = OpenAI()
    #   response = client.embeddings.create(
    #       model="text-embedding-3-small", input=text
    #   )
    #   embedding = response.data[0].embedding
    return hashlib.sha256(text.lower().strip().encode()).hexdigest()[:16]


# --- In-memory cache for demonstration (swap for Redis in production) ---
_cache: dict[str, str] = {}


def cached_completion(prompt: str) -> tuple[str, int]:
    """Check cache before calling the LLM.

    Returns (response_text, token_count).  token_count == 0 on cache hit.
    """
    cache_key = get_embedding_hash(prompt)

    # Check cache
    cached = _cache.get(cache_key)
    if cached:
        return cached, 0  # $0 cost!

    # Simulate an LLM call (replace with real client.chat.completions.create)
    result = f"[LLM response for: {prompt[:60]}]"
    tokens = len(prompt.split()) + len(result.split())

    # Cache the result
    _cache[cache_key] = result

    return result, tokens


# --- Demo ---
result1, cost1 = cached_completion("What is Python?")
print(f"Query 1 - tokens used: {cost1}, response: {result1}")

result2, cost2 = cached_completion("What is Python?")  # exact match → cache hit
print(f"Query 2 - tokens used: {cost2}, response: {result2}")

print(f"\nCache entries: {len(_cache)}")

### Production semantic caching with Redis

```python
import redis
from openai import OpenAI

client = OpenAI()
cache = redis.Redis(host="localhost", port=6379, db=0)

def cached_completion(prompt: str, ttl_seconds: int = 3600):
    cache_key = get_embedding_hash(prompt)
    cached = cache.get(cache_key)
    if cached:
        return cached.decode(), 0

    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[{"role": "user", "content": prompt}],
    )
    result = response.choices[0].message.content
    tokens = response.usage.total_tokens
    cache.setex(cache_key, ttl_seconds, result)
    return result, tokens
```

---

## 2. Batch Processing

Process many prompts concurrently rather than one at a time.
This can deliver **5-20x throughput** improvement depending on API rate limits.

In [ ]:
import asyncio
import time


async def _mock_llm_call(prompt: str) -> str:
    """Simulate an LLM API call with ~100ms latency."""
    await asyncio.sleep(0.1)
    return f"[Response for: {prompt[:40]}]"


async def process_batch(prompts: list[str], batch_size: int = 10) -> list[str]:
    """Process prompts in concurrent batches."""
    results: list[str] = []

    for i in range(0, len(prompts), batch_size):
        batch = prompts[i : i + batch_size]
        tasks = [_mock_llm_call(p) for p in batch]
        responses = await asyncio.gather(*tasks)
        results.extend(responses)

    return results


# --- Demo ---
prompts = [f"Explain concept #{i}" for i in range(20)]

# Sequential baseline
start = time.perf_counter()
sequential = []
for p in prompts:
    sequential.append(asyncio.get_event_loop().run_until_complete(_mock_llm_call(p)))
seq_time = time.perf_counter() - start

# Batched
start = time.perf_counter()
batched = asyncio.get_event_loop().run_until_complete(process_batch(prompts, batch_size=10))
batch_time = time.perf_counter() - start

print(f"Sequential: {seq_time:.2f}s  ({len(prompts)} prompts)")
print(f"Batched:    {batch_time:.2f}s  ({len(prompts)} prompts)")
print(f"Speedup:    {seq_time / batch_time:.1f}x")

---

## 3. Prompt Compression

Reducing token count directly reduces cost. Concise prompts also tend to reduce latency.

In [ ]:
def estimate_tokens(text: str) -> int:
    """Rough token estimate (~4 chars per token for English)."""
    return max(1, len(text) // 4)


# --- Verbose vs concise ---
verbose_prompt = """
I would like you to please help me understand the following concept.
Could you please explain it in a way that is easy to understand?
I am particularly interested in learning about the main points.
Please make sure to cover all the important aspects.
"""

concise_prompt = "Explain [concept] clearly. Cover main points."

verbose_tokens = estimate_tokens(verbose_prompt)
concise_tokens = estimate_tokens(concise_prompt)
savings_pct = (1 - concise_tokens / verbose_tokens) * 100

print(f"Verbose: ~{verbose_tokens} tokens")
print(f"Concise: ~{concise_tokens} tokens")
print(f"Savings: ~{savings_pct:.0f}%")

print("\n--- Compression techniques ---")
techniques = [
    ("Remove filler words", '"Please kindly help me" → "Help me"'),
    ("Use abbreviations", '"Natural Language Processing" → "NLP"'),
    ("Remove redundancy", '"explain and describe" → "explain"'),
    ("Use structured format", '"In bullet points" (let format do the work)'),
    ("System prompt for style", "Set tone once, don't repeat per message"),
]
for name, example in techniques:
    print(f"  • {name}: {example}")

---

## 4. Model Fallback (Cheap → Expensive)

Route simple queries to cheap models and reserve expensive models for complex tasks.

> **See also:** `09_llm_infrastructure.ipynb` for production-grade fallback routing with LiteLLM.

In [ ]:
# Pricing reference (per 1M tokens blended, April 2026)
# Source: artificialanalysis.ai
MODEL_TIERS = {
    "simple":  {"model": "gpt-5.4-nano",        "cost_per_1m": 0.46,  "speed_tok_s": 167},
    "medium":  {"model": "gpt-5.4-mini",         "cost_per_1m": 1.69,  "speed_tok_s": 146},
    "complex": {"model": "gpt-5.4",              "cost_per_1m": 5.63,  "speed_tok_s": 80},
    "budget":  {"model": "deepseek-v4-flash",     "cost_per_1m": 0.17,  "speed_tok_s": 83},
    "open":    {"model": "gpt-oss-120b",          "cost_per_1m": 0.26,  "speed_tok_s": 209},
}

# Full pricing reference for routing decisions (April 2026)
FULL_PRICING = {
    "gpt-5.5-xhigh":         {"cost_per_1m": 11.25, "intelligence": 60, "speed": 72},
    "claude-opus-4.7-max":    {"cost_per_1m": 10.00, "intelligence": 57, "speed": 48},
    "gemini-3.1-pro":         {"cost_per_1m":  4.50, "intelligence": 57, "speed": 119},
    "gpt-5.4-xhigh":         {"cost_per_1m":  5.63, "intelligence": 57, "speed": 80},
    "claude-sonnet-4.6-max":  {"cost_per_1m":  6.00, "intelligence": 52, "speed": 58},
    "gpt-5.4-mini-xhigh":    {"cost_per_1m":  1.69, "intelligence": 49, "speed": 146},
    "gemini-3-flash":         {"cost_per_1m":  1.13, "intelligence": 46, "speed": 158},
    "gpt-5.4-nano-xhigh":    {"cost_per_1m":  0.46, "intelligence": 44, "speed": 167},
    "deepseek-v4-flash":      {"cost_per_1m":  0.17, "intelligence": 47, "speed": 83},
    "gpt-oss-120b":           {"cost_per_1m":  0.26, "intelligence": 33, "speed": 209},
    "gpt-oss-20b":            {"cost_per_1m":  0.10, "intelligence": 24, "speed": 273},
    "claude-4.5-haiku":       {"cost_per_1m":  2.00, "intelligence": 37, "speed": 98},
}

COMPLEX_KEYWORDS = {"analyze", "compare", "design", "code", "debug", "architect"}


def estimate_complexity(prompt: str) -> str:
    """Heuristic complexity estimation."""
    prompt_lower = prompt.lower()
    if any(kw in prompt_lower for kw in COMPLEX_KEYWORDS):
        return "complex"
    if len(prompt.split()) > 100:
        return "medium"
    return "simple"


def route_request(prompt: str) -> dict:
    """Route to cheapest adequate model."""
    tier = estimate_complexity(prompt)
    chosen = MODEL_TIERS[tier]
    print(f"Complexity: {tier} -> Model: {chosen['model']} (${chosen['cost_per_1m']}/1M, {chosen['speed_tok_s']} tok/s)")
    return chosen


# Demo routing
for prompt in [
    "What is the capital of France?",
    "Summarize this 2000-word article about quantum computing",
    "Architect a microservices system for a real-time trading platform",
]:
    route_request(prompt)

print()
print("Full pricing reference (artificialanalysis.ai, April 2026):")
print(f"{'Model':<28} {'$/1M':>6} {'Intelligence':>13} {'Speed':>8}")
print("-" * 60)
for model, info in sorted(FULL_PRICING.items(), key=lambda x: x[1]['cost_per_1m']):
    print(f"{model:<28} ${info['cost_per_1m']:>5.2f} {info['intelligence']:>13} {info['speed']:>6} t/s")

---

## 5. Cost Optimization Reference (April 2026)

### Token Reduction Techniques

| Technique | Savings | Effort |
|-----------|---------|--------|
| Remove filler words | 10-20% | Low |
| Use abbreviations | 5-15% | Low |
| Compress examples | 20-40% | Medium |
| Smaller model | 50-95% | Medium |
| Semantic caching | 60-90% | High |
| Fine-tuned model | 50-80% | High |

### Model Selection Guide (April 2026)

```
Simple tasks (FAQ, classification):
└─ GPT-5.4 nano ($0.46/1M), gpt-oss-20B ($0.10/1M), or local Qwen3.5 4B

Medium tasks (summarization, extraction):
└─ GPT-5.4 mini ($1.69/1M), Gemini 3 Flash ($1.13/1M), DeepSeek V4 Flash ($0.17/1M)

Complex tasks (analysis, coding, reasoning):
└─ GPT-5.4 ($5.63/1M), Gemini 3.1 Pro ($4.50/1M), Claude Sonnet 4.6 ($6.00/1M)

Maximum quality (research, agents, highest stakes):
└─ GPT-5.5 ($11.25/1M), Claude Opus 4.7 ($10.00/1M)

Budget complex tasks:
└─ DeepSeek V4 Flash ($0.17/1M, Intelligence Index 47!)
```

### Cost Savings by Strategy

| Strategy | Example | Savings vs GPT-5.5 |
|----------|---------|-------------------|
| Use Gemini 3.1 Pro instead | Same quality tier (57 vs 60) | **60%** ($4.50 vs $11.25) |
| Use DeepSeek V4 Flash | Good quality (47) at budget price | **98.5%** ($0.17 vs $11.25) |
| Use gpt-oss-120B | Decent quality (33), blazing fast | **97.7%** ($0.26 vs $11.25) |
| Semantic caching | Cache frequent queries | **60-90%** of any model |
| Model routing | Route by complexity | **50-70%** mixed workloads |

*Pricing from [artificialanalysis.ai](https://artificialanalysis.ai/leaderboards/models) - April 2026*

---

---

## 6. Production Monitoring

In [ ]:
import functools
import time
from dataclasses import dataclass, field


@dataclass
class LLMMetrics:
    """Metrics for a single LLM invocation."""

    model: str
    tokens_input: int
    tokens_output: int
    latency_ms: float
    cost_usd: float
    cache_hit: bool
    error: str | None = None


# Simple in-memory metrics store (swap for Prometheus / Datadog in prod)
_metrics_log: list[LLMMetrics] = []


def track_llm_call(func):
    """Decorator to track LLM metrics."""

    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            result, meta = func(*args, **kwargs)
            _metrics_log.append(
                LLMMetrics(
                    model=meta.get("model", "unknown"),
                    tokens_input=meta.get("tokens_in", 0),
                    tokens_output=meta.get("tokens_out", 0),
                    latency_ms=(time.perf_counter() - start) * 1000,
                    cost_usd=meta.get("cost", 0.0),
                    cache_hit=meta.get("cache_hit", False),
                )
            )
            return result
        except Exception as exc:
            _metrics_log.append(
                LLMMetrics(
                    model="unknown",
                    tokens_input=0,
                    tokens_output=0,
                    latency_ms=(time.perf_counter() - start) * 1000,
                    cost_usd=0.0,
                    cache_hit=False,
                    error=str(exc),
                )
            )
            raise

    return wrapper


# --- Demo usage ---
@track_llm_call
def my_llm_call(prompt: str):
    """Example tracked function."""
    time.sleep(0.05)  # simulate latency
    return f"Answer to: {prompt[:30]}", {
        "model": "gpt-4.1-mini",
        "tokens_in": len(prompt.split()),
        "tokens_out": 25,
        "cost": 0.00003,
        "cache_hit": False,
    }


# Fire a few calls
for i in range(5):
    my_llm_call(f"Question number {i}")

# Report
total_cost = sum(m.cost_usd for m in _metrics_log)
avg_latency = sum(m.latency_ms for m in _metrics_log) / len(_metrics_log)
total_tokens = sum(m.tokens_input + m.tokens_output for m in _metrics_log)

print(f"Calls:         {len(_metrics_log)}")
print(f"Total tokens:  {total_tokens}")
print(f"Avg latency:   {avg_latency:.1f}ms")
print(f"Total cost:    ${total_cost:.5f}")

---

## 7. Best Practices

### Cost Optimization

**DO**
- Track all token usage
- Cache aggressively
- Use cheapest capable model
- Compress prompts
- Batch similar requests
- Set budget alerts
- Review costs weekly

**DON'T**
- Use the most expensive model for everything
- Ignore caching
- Process one-by-one
- Keep verbose prompts
- Skip monitoring
- Forget rate limits

### Performance Optimization

**DO**
- Stream responses
- Use async/parallel
- Implement timeouts
- Add retries with backoff
- Monitor latency
- Use CDN for static content
- Prefetch common queries

**DON'T**
- Block on LLM calls
- Ignore streaming
- Skip error handling
- Use synchronous code
- Forget timeout limits

---

## Resources

### Tools

- [LangSmith](https://www.langchain.com/langsmith) - LLM monitoring
- [Helicone](https://www.helicone.ai/) - LLM observability
- [Portkey](https://portkey.ai/) - Gateway & caching
- [LiteLLM](https://github.com/BerriAI/litellm) - Unified interface

### Caching Solutions

- Redis (in-memory)
- GPTCache (semantic)
- Momento (managed)
- Upstash (serverless)

---

## Optimization Checklist

- [ ] Track all token usage
- [ ] Implement semantic caching
- [ ] Use appropriate model for task
- [ ] Compress prompts (remove filler)
- [ ] Batch similar requests
- [ ] Set up monitoring dashboard
- [ ] Configure cost alerts
- [ ] Test fallback strategies
- [ ] Implement rate limiting
- [ ] Monitor cache hit rate
- [ ] Review costs monthly
- [ ] Optimize based on metrics

---

## Expected Savings

**Well-optimized LLM application:**

| Technique | Savings |
|-----------|--------|
| Caching | 60-80% cost reduction |
| Model selection | 50-70% cost reduction |
| Prompt optimization | 10-20% cost reduction |
| Streaming | 3-10x latency improvement |
| Batching | 5-20x throughput improvement |

**Total: 80-95% cost reduction** possible when all techniques are applied!

---

**Start optimizing:** Track your current costs first, then apply techniques incrementally.

**Measure everything:** You can't optimize what you don't measure.